# BASTION: Simulated Data Example

This notebook demonstrates BASTION on a **controlled simulation study**, where the true data-generating process is known.  It reproduces the core results from Section 5 of:

> Cho, J. B. & Matteson, D. S. (2026). *BASTION: A Bayesian Framework for Trend and Seasonality Decomposition.* arXiv:2601.18052.

## What is BASTION?

BASTION (Bayesian Adaptive Seasonality and Trend DecompositION) decomposes a univariate time series into:

- **Trend** — locally smooth, piecewise-linear, estimated with a global-local horseshoe prior on second differences.
- **Seasonal components** — one per specified period *k*, each penalised by second-differencing (first *k* steps) and seasonal differencing thereafter.
- **Outliers** (optional) — sparse additive spikes modelled with a horseshoe+ prior.
- **Remainder** — i.i.d. Gaussian noise, optionally with stochastic volatility.

Posterior inference is carried out with a **Gibbs sampler** whose cost per iteration is O(N) (Rue 2001 sparse precision sampler), making it practical for thousands of observations.

## This example

We generate a length-500 series with:
- a **piecewise-linear trend** with four segments and random slopes,
- **two seasonal components** (periods 7 and 30) defined as Fourier pairs,
- **three injected outliers**, and
- i.i.d. Gaussian noise.

We then fit BASTION and inspect:
1. the estimated signal (trend + seasonality) overlaid on the data,
2. the recovered trend against the truth with 95% posterior credible bands,
3. the individual seasonal components,
4. a simulation-average metrics table (Table 2 from the paper) placing these results in context.

In [ ]:
import os, sys, warnings
import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION

print(f'pybastion loaded from: {PROJECT_DIR}')

## 1. Generate the Simulated Series

The data-generating process (DGP 1 in the paper, Table 1) uses:
- Piecewise-linear trend with random slopes drawn from U(-20, 20) x 0.04.
- Two Fourier-pair seasonal components: S7 and S30.
- Additive Gaussian noise sigma = 1.
- Three point outliers with magnitude +/-U(15, 25).

In [ ]:
def generate_T(n=500, seed=4):
    """Piecewise-linear trend with 4 segments and random slopes."""
    rng = np.random.default_rng(seed)
    slopes = rng.uniform(-20, 20, 4) * 0.04
    bp = [0, 125, 250, 375, 500]
    T = np.zeros(n)
    for i in range(4):
        seg = np.arange(bp[i], bp[i + 1])
        T[seg] = T[bp[i]] + slopes[i] * np.arange(len(seg))
    return T

def gen_sim(n=500, seed=4):
    rng = np.random.default_rng(seed)
    t   = np.arange(1, n + 1)
    T   = generate_T(n, seed=seed)
    S7  = 4 * np.sin(2 * np.pi * t / 7)  + 4 * np.cos(2 * np.pi * t / 7)
    S30 = 5 * np.sin(2 * np.pi * t / 30) + 5 * np.cos(2 * np.pi * t / 30)
    eps = rng.normal(0, 1, n)
    y   = T + S7 + S30 + eps
    out_idx = [50, 200, 400]
    for i in out_idx:
        y[i] += rng.choice([-1, 1]) * rng.uniform(15, 25)
    return {'y': y, 'T': T, 'S7': S7, 'S30': S30, 'outliers': out_idx}

sim = gen_sim(n=500, seed=4)
t   = np.arange(1, 501)

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(t, sim['y'], s=6, color='black', alpha=0.6, label='Observed y')
ax.plot(t, sim['T'] + sim['S7'] + sim['S30'],
        linewidth=1.2, color='steelblue', label='True signal (T+S)')
ax.plot(t, sim['T'], linewidth=1.2, color='navy', linestyle='--', label='True trend T')
for i in sim['outliers']:
    ax.axvline(i + 1, color='red', linestyle=':', linewidth=1.0, alpha=0.7)
ax.set_xlabel('Time step'); ax.set_ylabel('Value')
ax.set_title('Simulated series -- true components shown')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
print(f'Series length: {len(sim["y"])}, outlier positions: {sim["outliers"]}')

## 2. Fit BASTION

**Key parameters:**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `Ks` | `[7, 30]` | Seasonal periods to estimate |
| `Outlier` | `True` | Enable explicit outlier component |
| `obsSV` | `'const'` | Constant observation variance (no stochastic volatility) |
| `nsave` | 2 000 | Post-burn-in draws to keep |
| `nburn` | 5 000 | Burn-in iterations |
| `nskip` | 4 | Thinning -- keep 1 in every 5 draws |
| `nchains` | 2 | Independent chains (run sequentially) |

Total MCMC steps per chain: `nburn + (nskip+1) x nsave` = 5 000 + 5 x 2 000 = **15 000**.  
Expected runtime: **~30-60 min** on a typical laptop.  
For a quick preview substitute `nsave=200, nburn=500, nskip=1, nchains=1` (~3 min).

In [ ]:
result  = fit_BASTION(
    sim['y'],
    Ks=[7, 30],
    Outlier=True,
    cl=0.95,
    obsSV='const',
    nsave=2000,
    nburn=5000,
    nskip=4,
    nchains=2,
    seed=40,
)
summary = result['summary']
print('Estimated components:', [k for k in summary if k.endswith('_sum')])

## 3. Inspecting the Decomposition

### 3a. Signal and trend overlaid on the data

The posterior mean of `Signal` (trend + all seasonal components) is shown in black; the trend alone in red.  Red dotted lines mark the true outlier positions -- note how the horseshoe+ prior shrinks those time points while leaving the rest unaffected.

In [ ]:
trend_mean  = np.asarray(summary['Trend_sum']['Mean']).ravel()
trend_lo    = np.asarray(summary['Trend_sum']['CR_lower']).ravel()
trend_hi    = np.asarray(summary['Trend_sum']['CR_upper']).ravel()
signal_mean = np.asarray(summary['Signal_sum']['Mean']).ravel()
signal_lo   = np.asarray(summary['Signal_sum']['CR_lower']).ravel()
signal_hi   = np.asarray(summary['Signal_sum']['CR_upper']).ravel()
s7_mean     = np.asarray(summary['Seasonal7_sum']['Mean']).ravel()
s30_mean    = np.asarray(summary['Seasonal30_sum']['Mean']).ravel()

n = len(sim['y']); t = np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(t, sim['y'], s=6, color='black', alpha=0.5, zorder=1, label='Observed')
ax.fill_between(t, signal_lo, signal_hi, color='grey', alpha=0.3, zorder=2,
                label='95% credible band')
ax.plot(t, signal_mean, linewidth=1.5, color='black', zorder=3, label='Signal (T+S)')
ax.plot(t, trend_mean,  linewidth=1.5, color='red',   zorder=4, label='Trend')
for i in sim['outliers']:
    ax.axvline(i + 1, color='red', linestyle='--', linewidth=1.0, alpha=0.6)
ax.set_xlabel('Time step'); ax.set_ylabel('Value')
ax.set_title('BASTION: estimated signal and trend on simulated data')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'sim_signal.pdf'), dpi=150, bbox_inches='tight')
plt.show()

### 3b. Recovered trend vs truth

The grey band is the 95% posterior credible interval for the trend alone.  Because BASTION uses a global-local shrinkage prior on second differences, the trend adapts sharply at breakpoints while remaining smooth elsewhere.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(t, trend_lo, trend_hi, color='grey', alpha=0.35, label='95% credible band')
ax.plot(t, trend_mean, linewidth=1.5, color='black', label='BASTION posterior mean')
ax.plot(t, sim['T'],   linewidth=1.5, color='steelblue', linestyle='--',
        label='True trend')
ax.set_xlabel('Time step'); ax.set_ylabel('Value')
ax.set_title('Trend recovery: BASTION vs true trend')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'sim_trend.pdf'), dpi=150, bbox_inches='tight')
plt.show()

mse_trend  = float(np.mean((trend_mean  - sim['T']) ** 2))
mse_signal = float(np.mean((signal_mean - (sim['T'] + sim['S7'] + sim['S30'])) ** 2))
print(f'This run -- Trend MSE  : {mse_trend:.4f}')
print(f'This run -- Signal MSE : {mse_signal:.4f}')

### 3c. Seasonal components

The two seasonal components (periods 7 and 30) with 95% credible bands.

In [ ]:
s7_lo  = np.asarray(summary['Seasonal7_sum']['CR_lower']).ravel()
s7_hi  = np.asarray(summary['Seasonal7_sum']['CR_upper']).ravel()
s30_lo = np.asarray(summary['Seasonal30_sum']['CR_lower']).ravel()
s30_hi = np.asarray(summary['Seasonal30_sum']['CR_upper']).ravel()

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, mean, lo, hi, true_, label in [
    (axes[0], s7_mean,  s7_lo,  s7_hi,  sim['S7'],  'Period-7 seasonality'),
    (axes[1], s30_mean, s30_lo, s30_hi, sim['S30'], 'Period-30 seasonality'),
]:
    ax.fill_between(t, lo, hi, color='grey', alpha=0.35, label='95% credible band')
    ax.plot(t, mean,  linewidth=1.2, color='black',     label='BASTION estimate')
    ax.plot(t, true_, linewidth=1.2, color='steelblue', linestyle='--',
            label='True seasonal')
    ax.set_ylabel('Value'); ax.set_title(label); ax.legend(fontsize=8)
axes[-1].set_xlabel('Time step')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'sim_seasonality.pdf'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Simulation-Average Metrics (Cho & Matteson 2026, Table 2)

The table below reproduces accuracy results from the paper's 1 000-replicate simulation study (DGP 1: piecewise-linear trend, two Fourier seasonalities, constant noise, no outliers).  Competitor methods (TBATS, MSTL, STR) are not part of the `pybastion` package and are shown for reference only.

| Method | Signal MSE | Trend MSE | Seasonality MSE |
|--------|------------|-----------|----------------|
| TBATS  | 13.878 | 3.624 | 11.672 |
| MSTL   | 11.760 | 13.227 | 2.712 |
| STR    | 10.829 | 13.785 | 2.575 |
| **BASTION** | **0.376** | **0.581** | **0.536** |

*Source: Table 2, DGP 1 in Cho & Matteson (2026). Results are averages over 1 000 replications; lower is better.*

BASTION achieves the lowest MSE across all components by an order of magnitude, because it simultaneously estimates and shrinks both the trend and seasonal components while accounting for outliers.

## 5. Uncertainty Quantification (Cho & Matteson 2026, Table 3)

A key advantage of the Bayesian formulation is calibrated uncertainty quantification. The table compares 95% empirical coverage and average interval width between BASTION and STR (the only frequentist competitor that provides uncertainty estimates).

| Component | STR coverage | BASTION coverage | STR width | BASTION width |
|---|---|---|---|---|
| Signal | 0.799 | **0.998** | 4.048 | 5.306 |
| Trend | 0.616 | **0.970** | 1.934 | 1.993 |
| Seasonality | 0.815 | **0.999** | 2.114 | 3.314 |

*Source: Table 3, DGP 1 in Cho & Matteson (2026). Nominal level is 95%; higher coverage is better.*

BASTION consistently maintains coverage **above** the nominal level for all components, whereas STR substantially undercovers (especially for trend and seasonality), meaning its intervals are too narrow to be reliable.